# 1D CNN-A — Latent Space Clustering

Loads the trained model checkpoint, extracts latent vectors for every clean window,
then groups them with K-Means and visualises the clusters with t-SNE.

> **Prerequisites:** run `1dcnn_a_learn.ipynb` first — it trains the model and saves
> `model.pt` to `DATA_DIR / SYMBOL /`.

## 1. Imports

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # macOS: prevents libiomp/libomp conflict

from datetime import date

import httpx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

print(f"PyTorch {torch.__version__} | device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. Config

In [ ]:
from config import Config

cfg = Config()

# ── Override defaults here before running the rest of the notebook ────────────
# cfg.MAX_BARS       = None   # load all bars (~552k)
# cfg.EPOCHS         = 30     # full training run
# cfg.N_CLUSTERS     = 12     # try more/fewer clusters
# cfg.LATENT_DIM     = 64     # larger latent space
# cfg.N_SAMPLE       = 5_000  # render more windows in Section 9

# Expose all config fields as module-level names so every downstream cell
# can use SYMBOL, WINDOW_SIZE, LR, feature_cols, DEVICE, etc. unchanged.
globals().update(vars(cfg))

print(f"Symbol={SYMBOL}  Timeframe={TIMEFRAME}  {START_DATE} → {END_DATE}")
print("Using device:", DEVICE)

## 3. Fetch Data from Alpaca API
Calls the local `alpaca_api` FastAPI service (must be running: `uv run main.py`).
Fetches TSLA 1-minute bars and saves to both DB and CSV.

In [ ]:
if FETCH_DATA:
    params = {
        "symbols": SYMBOL,
        "timeframe": TIMEFRAME,
        "start": START_DATE,
        "end": END_DATE,
        "save_to": "db,csv",
    }
    with httpx.Client(timeout=None) as client:
        r = client.get(f"{API_BASE}/bars", params=params)
        r.raise_for_status()
        result = r.json()
    bars = result.get("data", {}).get("bars", {}).get(SYMBOL, [])
    print(f"Fetched {len(bars)} bars for {SYMBOL}")
    print("Saved:", result.get("saved"))
else:
    print("FETCH_DATA=False — skipping. Set True in Config to re-pull.")

## 4. Load Data

In [ ]:
csv_path = os.path.join(DATA_DIR, SYMBOL, f"{TIMEFRAME}.csv")
df = pd.read_csv(csv_path, parse_dates=["timestamp"], nrows=MAX_BARS)
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Loaded {len(df):,} bars from {df['timestamp'].min()} to {df['timestamp'].max()}")
max_bars_display = 'all' if MAX_BARS is None else f'{MAX_BARS:,}'
print(f"(MAX_BARS={max_bars_display})")
df[["timestamp", "open", "high", "low", "close", "volume"]].tail()

Check for:

Duplicate timestamps
Missing timestamps (gaps)
NaNs
Infinite values
Bad OHLC relationships (high < low, etc.)

Typical checks:

In [ ]:
df = df.drop_duplicates(subset=["timestamp"])
df = df.dropna()

df.isnull().sum()
df.duplicated(subset=["timestamp"]).sum()
df[["timestamp", "open", "high", "low", "close", "volume"]].tail()

## 5. Verify Time Continuity
A CNN assumes a consistent sequence.

Look for:

missing bars
duplicate bars
irregular spacing

If you're using 1-minute candles, every row should be exactly 1 minutes apart.

In [ ]:
delta = df["timestamp"].diff()
dt = pd.to_timedelta(delta).dt.total_seconds()
print("Average time delta (seconds):", dt.mean())
print("Time delta distribution (seconds):")
print(dt.describe())

## 6. Add Features

In [ ]:
# EMAs
df["ema_9"] = df["close"].ewm(span=9, adjust=False).mean()
df["ema_21"] = df["close"].ewm(span=21, adjust=False).mean()
df["ema_50"] = df["close"].ewm(span=50, adjust=False).mean()

# MACD components
df["macd_12"] = df["close"].ewm(span=12, adjust=False).mean()
df["macd_26"] = df["close"].ewm(span=26, adjust=False).mean()

# MACD line
df["macd"] = df["macd_12"] - df["macd_26"]

# MACD signal line (9 EMA of MACD)
df["macd_9"] = df["macd"].ewm(span=9, adjust=False).mean()

# MACD histogram (optional but commonly used)
df["macd_hist"] = df["macd"] - df["macd_9"]

# Candle details
df["body"] = df["close"] - df["open"]
df["upper_wick"] = df["high"] - df[["open", "close"]].max(axis=1)
df["lower_wick"] = df[["open", "close"]].min(axis=1) - df["low"]



# other
df["return"] = df["close"].pct_change()
df["vol_return"] = df["volume"].pct_change()
df["log_return"] = np.log(df["close"] / df["close"].shift(1))
df["volume_ratio"] = (
    df["volume"] /
    df["volume"].rolling(20).mean()
)

# display sample of new features
# df[["timestamp", "close", "ema_9", "ema_21", "ema_50", "macd", "macd_9", "macd_hist", "body", "upper_wick", "lower_wick", "return", "vol_return", "log_return", "volume_ratio"]].tail()

df[df["body"] != 0][
    [
        "timestamp",
        "close",
        "ema_9",
        "ema_21",
        "ema_50",
        "macd",
        "macd_9",
        "macd_hist",
        "body",
        "upper_wick",
        "lower_wick",
        "return",
        "vol_return",
        "log_return",
        "volume_ratio",
    ]
].tail()

## 6. Remove Initial NaNs

Feature engineering creates NaNs.

In [ ]:
df = df.dropna().reset_index(drop=True)

## 7. Scale Features

This is critical.

CNNs train poorly on:

close = 45000
volume = 10000000
return = 0.001

all mixed together.

StandardScaler

Most common:

In [ ]:
from sklearn.preprocessing import StandardScaler, RobustScaler

# feature_cols is defined in config.py — edit it there to change which features are used
# scaler = StandardScaler()
# df[feature_cols] = scaler.fit_transform(df[feature_cols])

scaler = RobustScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])
# Often better for financial data due to outliers

df[["timestamp", "open", "high", "low", "close", "volume", "ema_9", "ema_21", "ema_50", "macd", "macd_9", "macd_hist", "body", "upper_wick", "lower_wick", "return", "vol_return", "log_return", "volume_ratio"]].tail()

## 8. Create Fixed-Length Windows

A CNN does not ingest an entire dataframe.

It ingests samples.

In [ ]:
n_features = len(feature_cols)
print(f"Features ({n_features}):", feature_cols)
print("Data shape:", df[feature_cols].shape)

data = df[feature_cols].to_numpy(dtype=np.float32)

X_raw = np.lib.stride_tricks.sliding_window_view(
    data,
    window_shape=WINDOW_SIZE,
    axis=0
).transpose(0, 2, 1)   # → (N, WINDOW_SIZE, n_features)

print("X_raw shape:", X_raw.shape)

## 11. Filter Gap Windows
A window that spans an overnight or weekend gap mixes pre-gap and post-gap bars — the CNN would learn noise, not patterns. Any window whose 64-bar span crosses a gap > 5 minutes is dropped.

In [ ]:
diffs_sec = df["timestamp"].diff().dt.total_seconds().fillna(0).to_numpy()
gap_positions = np.where(diffs_sec > 300)[0]   # > 5 min between consecutive bars

valid_mask = np.ones(len(X_raw), dtype=bool)
for gp in gap_positions:
    lo = max(0, gp - WINDOW_SIZE + 1)
    hi = min(len(X_raw), gp + 1)
    valid_mask[lo:hi] = False

X_clean = X_raw[valid_mask]
print(f"Gap positions: {len(gap_positions)}")
print(f"Removed {(~valid_mask).sum():,} gap-spanning windows")
print(f"Clean windows: {X_clean.shape[0]:,}  shape: {X_clean.shape}")

## 13. Autoencoder Model
Encoder compresses `(batch, 14, 64)` → latent vector `(batch, LATENT_DIM)`.
Decoder reconstructs `(batch, 14, 64)` from the latent vector.
Training loss is reconstruction MSE — no labels needed.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, n_features, latent_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_features, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),                          # → (batch, 32, 32)
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),                          # → (batch, 64, 16)
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),                          # → (batch, 128, 8)
        )
        self.fc = nn.Linear(128 * 8, latent_dim)

    def forward(self, x):
        h = self.conv(x).flatten(1)
        return self.fc(h)


class Decoder(nn.Module):
    def __init__(self, n_features, latent_dim):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 128 * 8)
        self.deconv = nn.Sequential(
            nn.ConvTranspose1d(128, 64, kernel_size=4, stride=2, padding=1),   # → (batch, 64, 16)
            nn.ReLU(),
            nn.ConvTranspose1d(64, 32, kernel_size=4, stride=2, padding=1),    # → (batch, 32, 32)
            nn.ReLU(),
            nn.ConvTranspose1d(32, n_features, kernel_size=4, stride=2, padding=1),  # → (batch, 14, 64)
        )

    def forward(self, z):
        h = self.fc(z).view(z.size(0), 128, 8)
        return self.deconv(h)


class ConvAutoencoder(nn.Module):
    def __init__(self, n_features, latent_dim):
        super().__init__()
        self.encoder = Encoder(n_features, latent_dim)
        self.decoder = Decoder(n_features, latent_dim)

    def forward(self, x):
        return self.decoder(self.encoder(x))


model = ConvAutoencoder(n_features=n_features, latent_dim=LATENT_DIM).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,}")
print(model)

In [ ]:
# WindowDataset is needed by the latent-extraction DataLoader (Section 15)
class WindowDataset(Dataset):
    def __init__(self, X):
        self.X = X
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx]   # input == reconstruction target


### Load Model Checkpoint

In [ ]:
model_path = os.path.join(DATA_DIR, SYMBOL, "model.pt")

model = ConvAutoencoder(n_features=n_features, latent_dim=LATENT_DIM).to(DEVICE)
model.load_state_dict(torch.load(model_path, map_location=DEVICE))
model.eval()

print(f"Loaded  : {model_path}")
print(f"  Architecture  : ConvAutoencoder(n_features={n_features}, latent_dim={LATENT_DIM})")
print(f"  Input shape   : (batch, {n_features}, {WINDOW_SIZE})  — channels-first")
print(f"  Device        : {DEVICE}")
print(f"  Parameters    : {sum(p.numel() for p in model.parameters()):,}")

## 15. Extract Latent Vectors
Run every clean window through the encoder to get its compressed representation.

In [ ]:
all_loader = DataLoader(WindowDataset(
    torch.tensor(X_clean).permute(0, 2, 1)
), batch_size=BATCH_SIZE, shuffle=False)

model.eval()
Z_list = []
with torch.no_grad():
    for batch in all_loader:
        Z_list.append(model.encoder(batch.to(DEVICE)).cpu().numpy())

Z = np.concatenate(Z_list)   # (N_clean, LATENT_DIM)
print(f"Latent matrix Z: {Z.shape}")

## 16. Cluster & Visualise Patterns
K-Means groups windows by latent similarity. t-SNE projects the 32-dimensional latent space to 2D so we can see the clusters. Cluster centroid plots reveal what each discovered pattern looks like.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

# --- K-Means clustering ---
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init="auto")
labels = kmeans.fit_predict(Z)
print("Cluster sizes:", dict(zip(*np.unique(labels, return_counts=True))))

# --- t-SNE on a subsample (full dataset is too large for t-SNE) ---
# TSNE_SAMPLE is defined in config.py
idx = np.random.choice(len(Z), size=min(TSNE_SAMPLE, len(Z)), replace=False)
Z_2d = TSNE(n_components=2, random_state=42, perplexity=50).fit_transform(Z[idx])

plt.figure(figsize=(10, 7))
scatter = plt.scatter(Z_2d[:, 0], Z_2d[:, 1], c=labels[idx],
                      cmap="tab10", s=3, alpha=0.6)
plt.colorbar(scatter, label="Cluster")
plt.title(f"t-SNE of Latent Space ({TSNE_SAMPLE:,} sample) — {N_CLUSTERS} clusters")
plt.tight_layout(); plt.show()

In [ ]:
# --- Cluster centroid patterns (full-width, stacked) ---
# PLOT_FEATURES is defined in config.py
plot_idx = [feature_cols.index(f) for f in PLOT_FEATURES]

for cluster_id in range(N_CLUSTERS):
    mask = labels == cluster_id
    centroid_windows = X_clean[mask][:, :, plot_idx]   # (n_in_cluster, WINDOW_SIZE, 4)
    mean_window = centroid_windows.mean(axis=0)        # (WINDOW_SIZE, 4)

    fig, ax = plt.subplots(figsize=(18, 3))
    for i, fname in enumerate(PLOT_FEATURES):
        ax.plot(mean_window[:, i], label=fname, alpha=0.85)
    ax.set_title(f"Cluster {cluster_id}  (n={mask.sum():,})", fontsize=13)
    ax.legend(fontsize=9)
    ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")
    ax.set_xlabel("Bar (minutes)")
    plt.tight_layout()
    plt.show()

In [ ]:
from IPython.display import display, Markdown

_feature_descriptions = {
    "close":        "where {sym}'s price was, scaled so 0 = the median price over the whole dataset",
    "ema_9":        "a smoothed version of {sym}'s price over the last 9 minutes (reacts fast to moves)",
    "ema_21":       "a smoothed version of {sym}'s price over the last 21 minutes (slower trend)",
    "ema_50":       "a smoothed version of {sym}'s price over the last 50 minutes (longer-term trend)",
    "macd":         "momentum — positive = {sym} has been rising, negative = falling",
    "macd_9":       "signal line: a 9-period EMA of macd; crossovers are a classic buy/sell signal",
    "macd_hist":    "distance between macd and its signal line — growing = momentum is building",
    "volume_ratio": "how busy {sym} trading was compared to the 20-minute average (1.0 = normal)",
    "body":         "size and direction of the candle body (positive = bullish close, negative = bearish)",
    "upper_wick":   "how far price reached above the open/close during the bar",
    "lower_wick":   "how far price reached below the open/close during the bar",
    "return":       "percentage price change bar-over-bar",
    "vol_return":   "percentage volume change bar-over-bar",
    "log_return":   "log-scale price change — better for statistical analysis than raw returns",
}

_feature_lines = "\n".join(
    f"- **{f}** — {_feature_descriptions.get(f, f).format(sym=SYMBOL)}"
    for f in PLOT_FEATURES
)

_date_min = df["timestamp"].min().strftime("%b %Y")
_date_max = df["timestamp"].max().strftime("%b %Y")
_n_bars   = len(df)
_n_raw    = X_raw.shape[0]
_n_clean  = X_clean.shape[0]
_n_removed = _n_raw - _n_clean

display(Markdown(f"""
## 17. What You're Seeing

### The t-SNE Scatter Plot
Your {SYMBOL} 1-minute dataset spans {_date_min} to {_date_max} — that's **{_n_bars:,} bars**.
After slicing into overlapping {WINDOW_SIZE}-bar ({WINDOW_SIZE}-minute) windows, you have **{_n_raw:,} raw windows**.
Removing the **{_n_removed:,}** that straddle an overnight or weekend gap leaves **{_n_clean:,} clean windows**.

The autoencoder watched every one of those {SYMBOL} windows and gave each one a "fingerprint" —
{LATENT_DIM} numbers that summarise what was happening in those {WINDOW_SIZE} minutes of trading.

The problem is: {LATENT_DIM} numbers are impossible to visualise. So **t-SNE** squashes those {LATENT_DIM} numbers
down to just 2 (an x and y position on the chart) while trying to keep similar fingerprints close together.

**What the dots mean:**
- Each dot = one {WINDOW_SIZE}-minute window of {SYMBOL} trading
- Dots that are close together = windows that *felt* similar to the model
- The colour = which group (cluster) K-Means assigned it to

**What to look for:**
- Tight, well-separated blobs of colour = the model found genuinely distinct {SYMBOL} market behaviours
- A big smeared mess with no structure = the model hasn't learned much yet (try more epochs or a larger `LATENT_DIM` of {LATENT_DIM})

---

### The K-Means Clustering
K-Means is like a sorting machine. You told it: *"sort all {_n_clean:,} {SYMBOL} fingerprints into {N_CLUSTERS} piles"*
(`N_CLUSTERS = {N_CLUSTERS}`). It doesn't know anything about finance — it just puts fingerprints that are
numerically similar into the same pile.

Each pile = a **pattern type** the autoencoder found hiding in your {SYMBOL} data.
You chose {N_CLUSTERS}, but there's nothing magic about that number — try fewer or more and see if
the cluster centroid plots look more or less distinct.

---

### The Cluster Centroid Plots
These answer: *"What does a typical {SYMBOL} window in each cluster actually look like?"*

For each of the {N_CLUSTERS} clusters, we average together every {SYMBOL} window in that group and plot the result.
Think of it like averaging many photos of faces — the blurry average still shows you the general shape.

**What each line means:**
{_feature_lines}

**Example patterns you might see in {SYMBOL}:**
- A cluster where `close` and `ema_9` slope sharply upward → **strong uptrend**
- A cluster where `macd` crosses from negative to positive → **momentum reversal**
- A cluster where `volume_ratio` spikes while price is flat → **accumulation / distribution**
- A cluster where everything is flat and near zero → **dead/quiet market** (lunch hour, pre-market)

The model discovered these {N_CLUSTERS} groupings on its own — you never told it what a "trend" or "reversal" is.
"""))